In [36]:
import pandas as pd
import numpy as np

In [37]:
df = pd.read_csv("Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [38]:
df.shape

(10000, 14)

In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.1 MB


In [40]:
df.duplicated().sum()

np.int64(0)

In [41]:
df["Exited"].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

In [42]:
df.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='str')

In [43]:
df["Geography"].value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [44]:
df["Gender"].value_counts()

Gender
Male      5457
Female    4543
Name: count, dtype: int64

In [45]:
df.drop(columns=["RowNumber",
                 "CustomerId",
                 "Surname"],
                 inplace=True)

df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [46]:
df = pd.get_dummies(
    df,
    columns=["Geography", "Gender"],
    drop_first=True,
    dtype=int
)

df.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,0,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,1,0
2,502,42,8,159660.80,3,1,0,113931.57,1,0,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,1,0


In [47]:
from sklearn.model_selection import train_test_split

X = df.drop(columns= ["Exited"])
y = df["Exited"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42,
                                                    test_size=0.2)

In [48]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [49]:
import tensorflow


In [50]:
from keras.models import Sequential
from keras.layers import Input, Dense

model = Sequential()
model.add(Input(shape=(11,)))
model.add(Dense(11, activation="sigmoid"))
model.add(Dense(4, activation="sigmoid"))
model.add(Dense(1, activation="sigmoid"))

In [51]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 11)             │           132 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4)              │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 185 (740.00 B)

 Trainable params: 185 (740.00 B)

 Non-trainable params: 0 (0.00 B)

In [52]:
model.compile(loss="binary_crossentropy",
              optimizer="Adam")

In [53]:
model.fit(
    X_train_scaled,
    y_train,
    epochs=30
)

Epoch 1/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 739us/step - loss: 0.5039
Epoch 2/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 666us/step - loss: 0.4862
Epoch 3/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 737us/step - loss: 0.4638
Epoch 4/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 670us/step - loss: 0.4470
Epoch 5/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 673us/step - loss: 0.4377
Epoch 6/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 741us/step - loss: 0.4327
Epoch 7/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 696us/step - loss: 0.4300
Epoch 8/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 734us/step - loss: 0.4282
Epoch 9/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 670us/step - loss: 0.4266
Epoch 10/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 673us/step - loss: 0.4252
Epoch 11/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 677us/step - loss: 0.4239
Epoch 12/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 666us/step - loss: 0.4223
Epoch 13/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 734us/step - loss: 0.4206
Epoch 14/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 659us/step - loss: 0.4189
Epoch 15/30
250/250 ━━━━━━━━━

In [54]:
model.layers[1].get_weights()

[array([[-0.83364326, -1.1562382 , -0.7823614 ,  0.01622834],
        [-0.4318814 ,  0.74352694,  0.35152268,  0.6910022 ],
        [-0.80760497, -0.8081603 , -1.2234647 , -0.3903136 ],
        [ 1.3306993 ,  0.48644015,  1.4107732 ,  0.94315743],
        [-1.2722317 , -0.52127874, -1.2012105 , -1.361656  ],
        [-0.49882013, -0.6530809 , -0.31220347, -0.73317844],
        [ 1.2756596 ,  1.7982223 ,  1.6084051 ,  1.8003306 ],
        [-0.8325567 , -0.4437252 , -0.28173625, -0.8658772 ],
        [-0.19665985, -1.0248108 , -0.8491109 , -1.0922177 ],
        [ 0.63889396, -0.00523825,  0.10265192,  0.6106419 ],
        [ 0.34834537,  0.8762769 ,  0.56121856,  0.15993804]],
       dtype=float32),
 array([-0.06001296, -0.22245654, -0.16507742, -0.18701147], dtype=float32)]

In [55]:
model.layers[0].get_weights()

[array([[-7.48923942e-02, -1.87155783e-01, -2.00176403e-01,
          1.94028139e-01, -2.46374458e-01,  1.06253430e-01,
          2.64272932e-02, -2.75862124e-02,  8.05214643e-02,
         -2.25597829e-01,  5.24186715e-02],
        [ 1.86691082e+00,  3.81888539e-01,  6.55436695e-01,
         -2.65554333e+00,  2.66664124e+00,  1.47586599e-01,
          8.05688739e-01,  5.50001534e-03,  9.44062620e-02,
         -1.32620305e-01,  1.63969412e-01],
        [ 1.07774427e-02,  5.55258654e-02,  8.73410702e-03,
          3.06906626e-02, -3.60124707e-02, -4.44193035e-01,
          1.16517946e-01, -2.65669674e-01,  1.54265448e-01,
         -3.37968141e-01, -7.95496702e-02],
        [ 3.30042690e-01, -3.92212987e-01, -3.24499637e-01,
          7.21944049e-02, -1.66826501e-01,  3.13308030e-01,
          6.79862976e-01,  4.59623873e-01,  2.25803763e-01,
         -2.02255920e-01, -2.43173927e-01],
        [ 6.02559865e-01, -6.47305846e-01, -2.25088120e+00,
         -5.37465572e-01,  4.48318899e-01,  

In [56]:
y_log = model.predict(X_test_scaled)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 799us/step


In [57]:
y_pred = np.where(y_log>0.5, 1, 0)

In [58]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,
               y_pred)

0.846